# 02 Estimate: baselines and models on the validation folds, choice of the main model

**Purpose.** Answer the second half of the project question: how well can the credit spread over SOFR of a newly
originated private credit loan be predicted at the start of its quarter, and with which model? This notebook runs the
horse race of deck 16 / notebook 209: the two committed baselines and every course model on the **same four
time-ordered validation folds** with the **same metric**, tunes each model on these folds, and records the chosen main
model before any test prediction is made.

**Input.** `data/processed/loans.csv` (built by `01_preprocess.ipynb`), loaded with `code/modelling_helpers.py`.
Only rows with `split == "train"` (2023Q2 to 2025Q2) are used until the last steps.

**Output.**
- `data/processed/predictions_validation.csv`: one row per model and validation loan (`model, fold, loan_id,
  quarter, family, y_true, y_pred, abs_error_bps`). Notebook 03 uses the same columns for the test predictions.
- Later steps (added in the next work session): the race table and figure, `output/tables/model_choice.csv`, the test
  and robustness predictions.

**What is fixed** (CLAUDE.md, `docs/lecturer_feedback_and_commitments.md`, `docs/for_modelling.md`, D41, D45):
- Target `spread_pp` (percentage points); errors reported in basis points (x 100).
- Validation: expanding window by quarter, one-quarter gap (to predict quarter t, train on loans up to t-2 that were
  filed before the start of t): validation quarters 2024Q3, 2024Q4, 2025Q1, 2025Q2.
- Metrics: MAE in bps (main), share of loans within plus or minus 50 bps. The model is chosen on the MAE pooled over
  all validation loans; the per-fold MAEs show the spread (deck 16, rule 4: a simpler model is preferred when the gap
  is within the spread across the folds).
- Features: `mh.NUM` + `mh.CAT`; never interest rate, fair value, cost, PIK or floor. All learned preprocessing inside
  a scikit-learn pipeline fitted on the training fold only.
- The test quarters (2025Q3 to 2026Q2) are not used for any choice; this notebook computes no test error (D45).

**Steps.** 0 setup, 1 data and folds, 2 baselines. (Next session: 3 linear models, 4 trees, 5 neural net,
6 horse race, 7 optional features, 8 model choice, 9 test predictions, 10 robustness predictions.)
Every step ends with a check. Decisions are logged in `docs/decisions.md`.

## Step 0. Setup

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

ROOT = Path.cwd().parent if Path.cwd().name == "code" else Path.cwd()
sys.path.insert(0, str(ROOT / "code"))
import modelling_helpers as mh  # noqa: E402

PROC = ROOT / "data" / "processed"
FIG = ROOT / "output" / "figures"
TAB = ROOT / "output" / "tables"
SEED = 42                       # used for every model with randomness
np.random.seed(SEED)
print("project folder:", ROOT.name, "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__)

project folder: DSF_Project | pandas 3.1.1 | scikit-learn 1.9.1


## Step 1. Data and validation folds (training rows only)

**What.** We load the working dataset and keep only the training rows. `mh.time_cv_splits` builds four
expanding-window folds: each validation quarter is predicted by data that ends two quarters earlier and was filed
before the validation quarter starts.

**Why.** The test quarters stay untouched until the end (deck 08, deck 16 rule 1). The folds imitate how the model
would be used: at the start of a quarter, only filings that are already public can be used.

**Check.** Only training rows; fold sizes; for every fold the last training quarter is at most t-2 and every training
loan was filed before the validation quarter starts; no missing values in the features.

In [2]:
df = mh.load_data(PROC / "loans.csv")
train = df[df["split"] == "train"].reset_index(drop=True)
FEATURES = mh.NUM + mh.CAT
folds = list(mh.time_cv_splits(train))

rows = []
for k, (tr_idx, va_idx) in enumerate(folds):
    tr, va = train.iloc[tr_idx], train.iloc[va_idx]
    t = va["quarter"].iloc[0]
    assert va["quarter"].nunique() == 1
    assert tr["quarter"].max() <= t - 2                       # one-quarter gap
    assert (tr["filed_date"] < t.start_time).all()            # training loans public at the cutoff
    rows.append({"fold": k, "validation_quarter": str(t), "n_train": len(tr),
                 "train_quarters": f"{tr['quarter'].min()} to {tr['quarter'].max()}", "n_validation": len(va)})
fold_table = pd.DataFrame(rows)
print(fold_table.to_string(index=False))

# Check
assert (train["split"] == "train").all() and train["quarter"].max() == pd.Period("2025Q2")
assert train[FEATURES].notna().all().all()
print(f"training rows: {len(train)} (test rows not used: {(df['split'] == 'test').sum()}); "
      f"validation loans over the 4 folds: {fold_table['n_validation'].sum()}")

 fold validation_quarter  n_train   train_quarters  n_validation
    0             2024Q3      295 2023Q2 to 2024Q1           180
    1             2024Q4      447 2023Q2 to 2024Q2           205
    2             2025Q1      627 2023Q2 to 2024Q3           151
    3             2025Q2      830 2023Q2 to 2024Q4           139
training rows: 1122 (test rows not used: 583); validation loans over the 4 folds: 675


## Step 2. Baselines

**What.** The two baselines committed in our project description, both with the t-2 rule (`docs/for_modelling.md`):
1. **Market median (t-2):** the median spread of all new BDC loans first seen in quarter t-2 and filed before the
   start of t (the column `mkt_spread_t2_all`, built in notebook 01). One number per quarter for every loan.
2. **Sector x size median:** the median spread in the loan's sector and size tercile over the quarters t-5 to t-2
   (fallback: the sector median, then baseline 1 when a cell has fewer than 5 loans).

As a **reference only** (not a committed baseline) we add the mean spread of the training fold, the "mean" baseline
of deck 16. It shows how much is lost by ignoring that spreads fell over time.

**Why.** A model earns its place only if it beats the best simple rule on the same folds and metric (deck P1). A deal
team could compute baseline 1 from public filings in minutes.

**Check.** (a) Baseline 1 is constant within a quarter and equals the feature column; it is compared with the median
of the all-BDC loan file for the same quarter (a plausibility check: notebook 01 computes it on a slightly wider pool
of new loans). (b) One baseline-2 prediction is recomputed step by step with separate code.

In [3]:
def fold_frame(model_name, k, rows, y_pred):
    # Predictions of one model on one validation fold, in the common format (see mh.error_frame)
    return mh.error_frame(rows, y_pred).assign(model=model_name, fold=k)


val_preds = []          # every model and baseline appends its validation predictions here
for k, (tr_idx, va_idx) in enumerate(folds):
    tr, va = train.iloc[tr_idx], train.iloc[va_idx]
    val_preds.append(fold_frame("Baseline 1: market median t-2", k, va, mh.baseline_t2(va)))
    val_preds.append(fold_frame("Baseline 2: sector x size median", k, va, mh.baseline_sector_size(train, va)))
    val_preds.append(fold_frame("Reference: training mean", k, va, np.full(len(va), tr[mh.TARGET].mean())))
val = pd.concat(val_preds, ignore_index=True)


def race_table(val):
    # Pooled MAE over all validation loans (choice criterion, D45), the MAE of each fold, and the share within 50 bps
    pooled = val.groupby("model", sort=False).apply(mh.summarize_errors, include_groups=False)
    by_fold = val.groupby(["model", "fold"], sort=False)["abs_error_bps"].mean().unstack()
    by_fold.columns = [f"mae_fold{c}_{fold_table.loc[c, 'validation_quarter']}" for c in by_fold.columns]
    out = pooled.join(by_fold)
    out["fold_spread_bps"] = by_fold.max(axis=1) - by_fold.min(axis=1)
    return out.round({"mae_bps": 1, "share_within_50bps": 3, **{c: 1 for c in by_fold.columns}, "fold_spread_bps": 1})


print(race_table(val).to_string())

                                      n  mae_bps  share_within_50bps  mae_fold0_2024Q3  mae_fold1_2024Q4  mae_fold2_2025Q1  mae_fold3_2025Q2  fold_spread_bps
model                                                                                                                                                        
Baseline 1: market median t-2     675.0     78.6               0.584              77.9              76.3              85.4              75.4             10.0
Baseline 2: sector x size median  675.0     98.3               0.385             115.8              99.3              95.5              77.2             38.6
Reference: training mean          675.0     96.0               0.409             112.0              93.1              96.4              79.2             32.7


In [4]:
# Check (a): baseline 1 is the column mkt_spread_t2_all, one value per quarter, known at the cutoff
per_q = train.groupby("quarter")["mkt_spread_t2_all"].nunique()
assert (per_q == 1).all()
all_bdc = mh.load_data(PROC / "loans_all_bdc.csv")
check = []
for t in fold_table["validation_quarter"].map(pd.Period):
    ref = all_bdc[(all_bdc["quarter"] == t - 2) & (all_bdc["filed_date"] < t.start_time)]
    check.append({"quarter": str(t), "baseline_1": train.loc[train["quarter"] == t, "mkt_spread_t2_all"].iloc[0],
                  "median_all_bdc_file_t-2": ref["spread_pp"].median(), "n_loans_t-2": len(ref)})
print(pd.DataFrame(check).to_string(index=False))

# Check (b): recompute baseline 2 for every validation loan with separate code (a loop over loans), including the
# fallbacks, and count which rule produced each prediction
by_hand, source = [], []
for k, (tr_idx, va_idx) in enumerate(folds):
    va = train.iloc[va_idx]
    t = va["quarter"].iloc[0]
    window = train[(train["quarter"] >= t - 5) & (train["quarter"] <= t - 2) & (train["filed_date"] < t.start_time)]
    cut1, cut2 = np.quantile(window["log_principal"], [1 / 3, 2 / 3])
    size = lambda x: 0 if x < cut1 else (1 if x < cut2 else 2)
    window_size = window["log_principal"].map(size)
    for _, loan in va.iterrows():
        same_sector = window[window["sector"] == loan["sector"]]
        cell = same_sector[window_size[same_sector.index] == size(loan["log_principal"])]
        if len(cell) >= 5:
            by_hand.append(cell["spread_pp"].median()); source.append("sector x size cell")
        elif len(same_sector) >= 5:
            by_hand.append(same_sector["spread_pp"].median()); source.append("fallback: sector median")
        else:
            by_hand.append(loan["mkt_spread_t2_all"]); source.append("fallback: baseline 1")
helper = val.loc[val["model"] == "Baseline 2: sector x size median", "y_pred"].to_numpy()
assert np.allclose(np.array(by_hand), helper)
print("baseline 2 recomputed by hand for all", len(helper), "validation loans: identical to the helper")
print(pd.Series(source).value_counts().to_string())

quarter  baseline_1  median_all_bdc_file_t-2  n_loans_t-2
 2024Q3        5.00                     5.00          547
 2024Q4        5.00                     5.00          671
 2025Q1        5.00                     5.00          650
 2025Q2        4.75                     4.75          728


baseline 2 recomputed by hand for all 675 validation loans: identical to the helper
sector x size cell         634
fallback: sector median     33
fallback: baseline 1         8


In [5]:
# Save the validation predictions so far (overwritten when the models are added)
val.to_csv(PROC / "predictions_validation.csv", index=False)
assert val.groupby("model").size().eq(fold_table["n_validation"].sum()).all()   # every model predicts every loan once
print("written: data/processed/predictions_validation.csv,", len(val), "rows,", val["model"].nunique(), "models")

written: data/processed/predictions_validation.csv, 2025 rows, 3 models
